In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install datasets huggingface_hub
!pip install transformers torch datasets
!pip install torch torchvision datasets transformers -q

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, ResNetForImageClassification
from datasets import load_dataset


In [ ]:
MODEL_SAVE_PATH = "/content/drive/My Drive/stanford_dogs_resnet.pth"

learning_rate = 0.0001
batch_size = 32
num_epochs = 10
num_classes = 120
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")


In [ ]:
print("Loading dataset...")
dataset = load_dataset("amaye15/stanford-dogs")
print(dataset)

class CustomDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        image = item['pixel_values']
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
from datasets import DatasetDict

train_valid_split = dataset['train'].train_test_split(test_size=0.2)
dataset = DatasetDict({
    'train': train_valid_split['train'],
    'validation': train_valid_split['test']
})


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

])

train_dataset = CustomDataset(dataset['train'], transform=transform)
val_dataset = CustomDataset(dataset['validation'], transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training samples: {len(train_loader.dataset)}")
print(f"Number of validation samples: {len(val_loader.dataset)}")

In [ ]:

print("Loading pretrained ResNet...")
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
resnet_model = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")

print("Modifying classification head...")
input_features = resnet_model.classifier[1].in_features
resnet_model.classifier[1] = nn.Linear(input_features, num_classes)


resnet_model = resnet_model.to(device)

print("Model loaded and modified.")


In [ ]:
print(resnet_model)

In [ ]:
for param in resnet_model.parameters():
    param.requires_grad = False

for param in resnet_model.classifier.parameters():
    param.requires_grad = True

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=learning_rate)

print("Loss function optimizer done")

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    train_loss = 0.0

    for i, (inputs, labels) in enumerate(loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    return train_loss / len(loader)

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

def validate_epoch(model, loader, criterion, device):
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            loss = criterion(outputs.logits, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs.logits, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = 100 * correct / total

    # Calculate metrics
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=1)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=1)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=1)
    conf_matrix = confusion_matrix(all_labels, all_preds)

    return val_loss / len(loader), val_acc, precision, recall, f1, conf_matrix


#def validate_epoch(model, loader, criterion, device):
#    model.eval()
 #   val_loss, correct, total = 0.0, 0, 0
#
 #   with torch.no_grad():
  #      for inputs, labels in loader:
   #         inputs, labels = inputs.to(device), labels.to(device)
    #        outputs = model(inputs)
#
 #           loss = criterion(outputs.logits, labels)
  #          val_loss += loss.item()
#
 #           _, preds = torch.max(outputs.logits, 1)
  #          correct += (preds == labels).sum().item()
   #         total += labels.size(0)
#
 #   val_acc = 100 * correct / total
  #  return val_loss / len(loader), val_acc


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
class_names = ['Affenpinscher', 'Afghan hound', 'African hunting dog', 'Airedale', 'American Staffordshire terrier',  'Appenzeller', 'Australian terrier', 'Basenji', 'Basset', 'Beagle', 'Bedlington terrier',
 'Bernese mountain dog', 'Black-and-tan coonhound', 'Blenheim spaniel', 'Bloodhound', 'Bluetick',
 'Border collie', 'Border terrier', 'Borzoi', 'Boston bull', 'Bouvier des Flandres', 'Boxer',
 'Brabancon griffon', 'Briard', 'Brittany spaniel', 'Bull mastiff', 'Cairn', 'Cardigan',
 'Chesapeake Bay retriever', 'Chihuahua', 'Chow', 'Clumber', 'Cocker spaniel', 'Collie',
 'Curly-coated retriever', 'Dandie Dinmont', 'Dhole', 'Dingo', 'Doberman', 'English foxhound',
 'English setter', 'English springer', 'EntleBucher', 'Eskimo dog', 'Flat-coated retriever',
 'French bulldog', 'German shepherd', 'German short-haired pointer', 'Giant schnauzer',
 'Golden retriever', 'Gordon setter', 'Great Dane', 'Great Pyrenees', 'Greater Swiss Mountain dog',
 'Groenendael', 'Ibizan hound', 'Irish setter', 'Irish terrier', 'Irish water spaniel', 'Irish wolfhound',
 'Italian greyhound', 'Japanese spaniel', 'Keeshond', 'Kelpie', 'Kerry blue terrier', 'Komondor',
 'Kuvasz', 'Labrador retriever', 'Lakeland terrier', 'Leonberg', 'Lhasa', 'Malamute', 'Malinois',
 'Maltese dog', 'Mexican hairless', 'Miniature pinscher', 'Miniature poodle', 'Miniature schnauzer',
 'Newfoundland', 'Norfolk terrier', 'Norwegian elkhound', 'Norwich terrier', 'Old English sheepdog',
 'Otterhound', 'Papillon', 'Pekinese', 'Pembroke', 'Pomeranian', 'Pug', 'Redbone', 'Rhodesian ridgeback',
 'Rottweiler', 'Saint Bernard', 'Saluki', 'Samoyed', 'Schipperke', 'Scotch terrier', 'Scottish deerhound',
 'Sealyham terrier', 'Shetland sheepdog', 'Shih-Tzu', 'Siberian husky', 'Silky terrier',
 'Soft-coated wheaten terrier', 'Staffordshire bullterrier', 'Standard poodle', 'Standard schnauzer',
 'Sussex spaniel', 'Tibetan mastiff', 'Tibetan terrier', 'Toy poodle', 'Toy terrier', 'Vizsla',
 'Walker hound', 'Weimaraner', 'Welsh springer spaniel', 'West Highland white terrier', 'Whippet',
 'Wire-haired fox terrier', 'Yorkshire terrier']

def plot_confusion_matrix(conf_matrix, class_names, save_path):
    assert conf_matrix.shape[0] == conf_matrix.shape[1] == len(class_names), "Mismatch in dimensions"
    plt.figure(figsize=(30, 25))
    sns.heatmap(conf_matrix,annot=False,cmap='YlOrRd', xticklabels=class_names, yticklabels=class_names, cbar_kws={"shrink": 0.75})
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title('Confusion Matrix')
    plt.savefig(save_path)
    plt.show()
    plt.close()


In [ ]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    train_loss = train_epoch(resnet_model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, precision, recall, f1, conf_matrix = validate_epoch(resnet_model, val_loader, criterion, device)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%, Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1:.2f}")
    print(f"Confusion Matrix:\n{conf_matrix}")

    save_path = f'confusion_matrix_epoch_{epoch + 1}.png'
    plot_confusion_matrix(conf_matrix, class_names, save_path)

torch.save(resnet_model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved at {MODEL_SAVE_PATH}")


In [ ]:
for param in resnet_model.parameters():
    param.requires_grad = True

for param in resnet_model.classifier.parameters():
    param.requires_grad = True

for name, param in resnet_model.named_parameters():
    print(name, param.requires_grad)


In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.classifier(self.avg_pool(x).view(b, c)).view(b, c, 1, 1)
        max_out = self.classifier(self.max_pool(x).view(b, c)).view(b, c, 1, 1)
        out = avg_out + max_out
        return self.sigmoid(out) * x


In [ ]:
#Spatial

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size = (kernel_size, kernel_size), padding = (kernel_size // 2), bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(concat)
        return self.sigmoid(out) * x


In [ ]:
class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x  # Return only the processed tensor


In [ ]:
class ResNetWithCBAM(nn.Module):
    def __init__(self, base_model, num_classes):
        super(ResNetWithCBAM, self).__init__()

        self.resnet = base_model.resnet
        self.conv1 = self.resnet.embedder.embedder.convolution
        self.bn1 = self.resnet.embedder.embedder.normalization
        self.relu = self.resnet.embedder.embedder.activation
        self.maxpool = self.resnet.embedder.pooler

        self.layer1 = self.resnet.encoder.stages[0]  # ResNet Stage 1


        self.layer2 = self.resnet.encoder.stages[1]  # ResNet Stage 2
        self.layer3 = self.resnet.encoder.stages[2]  # ResNet Stage 3
        self.layer4 = self.resnet.encoder.stages[3]  # ResNet Stage 4

        self.cbam1 = CBAM(256)
        self.cbam2 = CBAM(512)

        self.avgpool = self.resnet.pooler
        in_features = base_model.classifier[-1].in_features
        self.classifier = nn.Linear(in_features, num_classes)



    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)

        x = self.cbam1(x)

        x = self.layer2(x)

        x = self.cbam2(x)
        x = self.layer3(x)


        x = self.layer4(x)


        x = self.avgpool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x


In [ ]:
resnet_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
resnet_model = resnet_model.to(device)
print("Model loaded")


In [ ]:
resnet_cbam = ResNetWithCBAM(resnet_model, num_classes=num_classes)
resnet_cbam.to(device)

In [ ]:
for param in resnet_cbam.conv1.parameters():
    param.requires_grad = True
for param in resnet_cbam.bn1.parameters():
    param.requires_grad = True

for param in resnet_cbam.relu.parameters():
    param.requires_grad = True
for param in resnet_cbam.maxpool.parameters():
    param.requires_grad = True

for param in resnet_cbam.layer1.parameters():
    param.requires_grad = True
for param in resnet_cbam.layer2.parameters():
    param.requires_grad = True

for param in resnet_cbam.layer3.parameters():
    param.requires_grad = True
for param in resnet_cbam.layer4.parameters():
    param.requires_grad = True
for param in resnet_cbam.classifier.parameters():
    param.requires_grad = True
for param in resnet_cbam.parameters():
     param.requires_grad = True

for name, param in resnet_cbam.named_parameters():
    print(name, param.requires_grad)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet_cbam.parameters()), lr=learning_rate)

print("Loss function optimizer done")

In [ ]:
def train_epoch_2(model, loader, optimizer, criterion, device):
    model.train()
    train_loss = 0.0

    for i, (inputs, labels) in enumerate(loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    return train_loss / len(loader)

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

def validate_epoch_2(model, loader, criterion, device):
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = 100 * correct / total

    # Calculate metrics
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=1)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=1)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=1)
    conf_matrix = confusion_matrix(all_labels, all_preds)

    return val_loss / len(loader), val_acc, precision, recall, f1, conf_matrix


#def validate_epoch(model, loader, criterion, device):
#    model.eval()
 #   val_loss, correct, total = 0.0, 0, 0
#
 #   with torch.no_grad():
  #      for inputs, labels in loader:
   #         inputs, labels = inputs.to(device), labels.to(device)
    #        outputs = model(inputs)
#
 #           loss = criterion(outputs.logits, labels)
  #          val_loss += loss.item()
#
 #           _, preds = torch.max(outputs.logits, 1)
  #          correct += (preds == labels).sum().item()
   #         total += labels.size(0)
#
 #   val_acc = 100 * correct / total
  #  return val_loss / len(loader), val_acc


In [ ]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    train_loss = train_epoch_2(resnet_cbam, train_loader, optimizer, criterion, device)
    val_loss, val_acc, precision, recall, f1, conf_matrix = validate_epoch_2(resnet_cbam, val_loader, criterion, device)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%, Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1:.2f}")
    print(f"Confusion Matrix:\n{conf_matrix}")

    save_path = f'confusion_matrix_epoch_{epoch + 1}.png'
    plot_confusion_matrix(conf_matrix, class_names, save_path)

torch.save(resnet_model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved at {MODEL_SAVE_PATH}")


In [ ]:
print(resnet_model)
# Before Resnet